In [1]:
import pandas as pd
import numpy as np
import torch
from pathlib import Path

Load the PRISM cell-line metadata

In [2]:
cell_info_path = "../data/raw/secondary-screen-cell-line-info.csv"

cell_info = pd.read_csv(cell_info_path)

print("Shape:", cell_info.shape)
print("Columns:", cell_info.columns.tolist())

Shape: (588, 7)
Columns: ['row_name', 'depmap_id', 'ccle_name', 'primary_tissue', 'secondary_tissue', 'tertiary_tissue', 'passed_str_profiling']


Select pancreatic cell lines

In [3]:
pancreatic_cells = cell_info[
    cell_info["primary_tissue"].astype(str).str.lower() == "pancreas"
].copy()

pancreatic_ids = pancreatic_cells["depmap_id"].tolist()

print("Pancreatic cell lines:", len(pancreatic_cells))
print(pancreatic_cells[["depmap_id", "ccle_name"]].to_string(index=False))

Pancreatic cell lines: 37
 depmap_id          ccle_name
ACH-000601  MIAPACA2_PANCREAS
ACH-000332      YAPC_PANCREAS
ACH-000178    HS766T_PANCREAS
ACH-000108       KP3_PANCREAS
ACH-000164     PANC1_PANCREAS
ACH-000599  PATU8902_PANCREAS
ACH-000347      QGP1_PANCREAS
ACH-000307       PK1_PANCREAS
ACH-000205      PK59_PANCREAS
ACH-000243      DANG_PANCREAS
ACH-000022 PATU8988S_PANCREAS
ACH-000155    SW1990_PANCREAS
ACH-000042  PANC0203_PANCREAS
ACH-000468     PK45H_PANCREAS
ACH-000270      HPAC_PANCREAS
ACH-000535     BXPC3_PANCREAS
ACH-000281       KP2_PANCREAS
ACH-000023 PATU8988T_PANCREAS
ACH-000652     SUIT2_PANCREAS
ACH-000320      PSN1_PANCREAS
ACH-000138    CFPAC1_PANCREAS
ACH-000118     HUPT3_PANCREAS
ACH-000235  PANC0403_PANCREAS
ACH-000266    SNU213_PANCREAS
ACH-000222     ASPC1_PANCREAS
ACH-000139  PANC0327_PANCREAS
ACH-000107    CAPAN2_PANCREAS
ACH-000502   TCCPAN2_PANCREAS
ACH-000517    SNU410_PANCREAS
ACH-000417  PANC0813_PANCREAS
ACH-000060  PANC1005_PANCREAS
ACH-000265    

Load GraphTCDR cell-line mapping

In [4]:
graphtcdr_cells = pd.read_csv(
    "../data/raw/cellLineName402.csv"
)

print("Shape:", graphtcdr_cells.shape)
print(graphtcdr_cells.head())

Shape: (402, 2)
   index           cellLineName
0      0     PATU8988S_PANCREAS
1      1          NCIH2110_LUNG
2      2  HCT15_LARGE_INTESTINE
3      3    SW780_URINARY_TRACT
4      4    CAL29_URINARY_TRACT


Match pancreatic cells with GraphTCDR

In [5]:
pancreatic_names = set(
    pancreatic_cells["ccle_name"]
    .astype(str)
    .str.upper()
)

graphtcdr_pancreatic = graphtcdr_cells[
    graphtcdr_cells["cellLineName"]
    .astype(str)
    .str.upper()
    .isin(pancreatic_names)
].copy()

print("Matched pancreatic cell lines:", len(graphtcdr_pancreatic))
print(
    graphtcdr_pancreatic[
        ["index", "cellLineName"]
    ].to_string(index=False)
)

Matched pancreatic cell lines: 28
 index       cellLineName
     0 PATU8988S_PANCREAS
    39  PANC0327_PANCREAS
    47     PANC1_PANCREAS
    68      PSN1_PANCREAS
    79     SUIT2_PANCREAS
    84    SNU213_PANCREAS
    85  PANC0813_PANCREAS
    91      DANG_PANCREAS
   103     HUPT3_PANCREAS
   123    CAPAN2_PANCREAS
   135     PK45H_PANCREAS
   160  PANC0403_PANCREAS
   165       L33_PANCREAS
   166    SNU410_PANCREAS
   179   TCCPAN2_PANCREAS
   234     BXPC3_PANCREAS
   237  PATU8902_PANCREAS
   266      QGP1_PANCREAS
   291    CFPAC1_PANCREAS
   294  PANC1005_PANCREAS
   313      HPAC_PANCREAS
   319    SW1990_PANCREAS
   321       PK1_PANCREAS
   333 PATU8988T_PANCREAS
   362     ASPC1_PANCREAS
   384      YAPC_PANCREAS
   386  PANC0203_PANCREAS
   400    HS766T_PANCREAS


Verify GraphTCDR index order

In [6]:
print(
    "Index order correct:",
    (graphtcdr_cells["index"].values == range(402)).all()
)

Index order correct: True


Get GraphTCDR pancreatic indices

In [7]:
pancreatic_indices = graphtcdr_pancreatic["index"].tolist()

print("Pancreatic indices:")
print(pancreatic_indices)

Pancreatic indices:
[0, 39, 47, 68, 79, 84, 85, 91, 103, 123, 135, 160, 165, 166, 179, 234, 237, 266, 291, 294, 313, 319, 321, 333, 362, 384, 386, 400]


Load all four GraphTCDR omics files

In [8]:
gene_expr = torch.load(
    "../data/raw/GeneExpressionFeature402.pt",
    weights_only=False
)

cnv = torch.load(
    "../data/raw/CNVFeature402.pt",
    weights_only=False
)

mirna = torch.load(
    "../data/raw/miRNAFeature402.pt",
    weights_only=False
)

methylation = torch.load(
    "../data/raw/MethylationFeature402.pt",
    weights_only=False
)

print("Gene expression:", gene_expr.shape)
print("CNV:", cnv.shape)
print("miRNA:", mirna.shape)
print("Methylation:", methylation.shape)

Gene expression: torch.Size([402, 20404])
CNV: torch.Size([402, 22582])
miRNA: torch.Size([402, 734])
Methylation: torch.Size([402, 20587])


Extract the 28 pancreatic cell lines

In [9]:
pancreatic_gene_expr = gene_expr[pancreatic_indices]
pancreatic_cnv = cnv[pancreatic_indices]
pancreatic_mirna = mirna[pancreatic_indices]
pancreatic_methylation = methylation[pancreatic_indices]

print("Pancreatic gene expression:", pancreatic_gene_expr.shape)
print("Pancreatic CNV:", pancreatic_cnv.shape)
print("Pancreatic miRNA:", pancreatic_mirna.shape)
print("Pancreatic methylation:", pancreatic_methylation.shape)

Pancreatic gene expression: torch.Size([28, 20404])
Pancreatic CNV: torch.Size([28, 22582])
Pancreatic miRNA: torch.Size([28, 734])
Pancreatic methylation: torch.Size([28, 20587])


Load PRISM response data

In [10]:
response_path = "../data/raw/secondary-screen-dose-response-curve-parameters.csv"

pancreatic_response = pd.read_csv(response_path)

print("Full PRISM response shape:", pancreatic_response.shape)

C:\Users\SREEKUTTY\AppData\Local\Temp\ipykernel_30420\829589429.py:3: DtypeWarning: Columns (14,15) have mixed types. Specify dtype option on import or set low_memory=False.
  pancreatic_response = pd.read_csv(response_path)


Full PRISM response shape: (701004, 20)


Create the 28-cell-line PRISM subset

In [11]:
pancreatic_response_28 = pancreatic_response[
    pancreatic_response["depmap_id"].isin(
        pancreatic_cells[
            pancreatic_cells["ccle_name"].isin(
                graphtcdr_pancreatic["cellLineName"]
            )
        ]["depmap_id"]
    )
].copy()

print("PRISM response records:", len(pancreatic_response_28))
print("Unique cell lines:", pancreatic_response_28["depmap_id"].nunique())
print("Unique drugs:", pancreatic_response_28["name"].nunique())

PRISM response records: 40119
Unique cell lines: 28
Unique drugs: 1448


IC50 availability

In [12]:
ic50_col = "ic50"

print("Total records:", len(pancreatic_response_28))
print(
    "IC50 available:",
    pancreatic_response_28[ic50_col].notna().sum()
)
print(
    "IC50 missing:",
    pancreatic_response_28[ic50_col].isna().sum()
)

Total records: 40119
IC50 available: 18913
IC50 missing: 21206


Prepare finite IC50 values

In [13]:
ic50_values = pd.to_numeric(
    pancreatic_response_28["ic50"],
    errors="coerce"
)

ic50_values = ic50_values[
    np.isfinite(ic50_values)
]

log_ic50 = np.log10(ic50_values)

print(log_ic50.describe())

count    18912.000000
mean         0.005375
std          3.085030
min       -110.746563
25%         -0.353090
50%          0.197369
75%          0.511461
max        287.762339
Name: ic50, dtype: float64


Tail + R² analysis

In [14]:
tail_check = pancreatic_response_28.copy()

tail_check["ic50"] = pd.to_numeric(
    tail_check["ic50"],
    errors="coerce"
)

tail_check = tail_check[
    np.isfinite(tail_check["ic50"])
].copy()

tail_check["log_ic50"] = np.log10(
    tail_check["ic50"]
)

print(
    tail_check.groupby(
        pd.cut(
            tail_check["log_ic50"],
            bins=[
                -np.inf, -3, -2, -1,
                0, 1, 2, 3, np.inf
            ]
        )
    )["r2"].agg(
        ["count", "median", "mean"]
    )
)

              count    median      mean
log_ic50                               
(-inf, -3.0]    225  0.108885  0.226843
(-3.0, -2.0]    428  0.766693  0.693731
(-2.0, -1.0]   1615  0.738775  0.690050
(-1.0, 0.0]    5240  0.608689  0.576298
(0.0, 1.0]    11082  0.411352  0.379779
(1.0, 2.0]      216  0.123736  0.156980
(2.0, 3.0]       34  0.001938 -0.421490
(3.0, inf]       72 -0.012675 -0.041088


C:\Users\SREEKUTTY\AppData\Local\Temp\ipykernel_30420\878657322.py:17: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  tail_check.groupby(


Extreme tail counts

In [15]:
print(
    "log10(IC50) < -3:",
    (tail_check["log_ic50"] < -3).sum()
)

print(
    "log10(IC50) > 3:",
    (tail_check["log_ic50"] > 3).sum()
)

print(
    "Total extreme:",
    (
        (tail_check["log_ic50"] < -3) |
        (tail_check["log_ic50"] > 3)
    ).sum()
)

log10(IC50) < -3: 225
log10(IC50) > 3: 72
Total extreme: 297


SMILES coverage

In [16]:
usable_response = tail_check.copy()

print(
    "Drugs with SMILES:",
    usable_response["smiles"].notna().sum()
)

print(
    "Missing SMILES:",
    usable_response["smiles"].isna().sum()
)

Drugs with SMILES: 18912
Missing SMILES: 0


Unique drugs and cell lines

In [17]:
print(
    "Unique drugs:",
    usable_response["name"].nunique()
)

print(
    "Unique cell lines:",
    usable_response["depmap_id"].nunique()
)

Unique drugs: 1228
Unique cell lines: 28


Response matrix

In [18]:
response_matrix = usable_response.pivot_table(
    index="depmap_id",
    columns="name",
    values="ic50",
    aggfunc="first"
)

observed = response_matrix.notna().sum().sum()
possible = (
    response_matrix.shape[0] *
    response_matrix.shape[1]
)

print("Matrix shape:", response_matrix.shape)
print("Observed IC50:", observed)
print("Possible combinations:", possible)

print(
    "Sparsity:",
    1 - observed / possible
)

Matrix shape: (28, 1228)
Observed IC50: 18037
Possible combinations: 34384
Sparsity: 0.4754246161005119
